# Import necessary libraries

In [ ]:
import uuid
import requests
import os
from langfuse.langchain import CallbackHandler
from langfuse import Langfuse, observe, get_client, propagate_attributes
from langgraph.graph import StateGraph, END
from groq import Groq
from duckduckgo_search import DDGS

# Load credentials and configuration from environment variables for Groq and Langfuse.
try:
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
    LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
    LANGFUSE_HOST = os.environ.get("LANGFUSE_BASE_URL", "https://api.langfuse.com")
    print(LANGFUSE_HOST)
except Exception as e:
    print(f"Error retrieving API keys from environment: {e}")
    raise ValueError("Ensure GROQ_API_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_PUBLIC_KEY, and LANGFUSE_BASE_URL are set in environment variables.")

# Debug: Print keys to verify retrieval (remove in production for security)

print("Debug: Verifying API keys retrieval...")
print(f"GROQ_API_KEY: {'Set' if GROQ_API_KEY else 'Not set'}")
print(f"LANGFUSE_PUBLIC_KEY: {'Set' if LANGFUSE_PUBLIC_KEY else 'Not set'}")
print(f"LANGFUSE_SECRET_KEY: {'Set' if LANGFUSE_SECRET_KEY else 'Not set'}")
print(f"LANGFUSE_HOST: {LANGFUSE_HOST if LANGFUSE_HOST else 'Not set'}")


# Validate that API keys are set
if not all([LANGFUSE_SECRET_KEY, LANGFUSE_PUBLIC_KEY, LANGFUSE_HOST, GROQ_API_KEY]):
    raise ValueError("One or more API keys are missing in Environment Variables.")

https://cloud.langfuse.com
Debug: Verifying API keys retrieval...
GROQ_API_KEY: Set
LANGFUSE_PUBLIC_KEY: Set
LANGFUSE_SECRET_KEY: Set
LANGFUSE_HOST: https://cloud.langfuse.com


# List available models to confirm valid model IDs

In [106]:
print("Listing available models from Groq API...")
try:
    url = "https://api.groq.com/openai/v1/models"
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    models = response.json()
    print("Available models:", [model['id'] for model in models['data']])
except Exception as e:
    print(f"Error listing models: {e}")
    raise

Listing available models from Groq API...
Available models: ['llama-3.1-8b-instant', 'openai/gpt-oss-safeguard-20b', 'meta-llama/llama-prompt-guard-2-22m', 'qwen/qwen3.6-27b', 'groq/compound-mini', 'meta-llama/llama-prompt-guard-2-86m', 'groq/compound', 'llama-3.3-70b-versatile', 'whisper-large-v3', 'whisper-large-v3-turbo', 'openai/gpt-oss-20b', 'allam-2-7b', 'canopylabs/orpheus-v1-english', 'openai/gpt-oss-120b', 'canopylabs/orpheus-arabic-saudi']


# Test Groq API key with a simple request

In [107]:
print("Testing Groq API key...")
try:
    groq_client = Groq(api_key=GROQ_API_KEY)
    test_response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "Hello"}],
        max_tokens=10
    )
    print("Groq API key test successful:", test_response.choices[0].message.content)
except Exception as e:
    print(f"Groq API key test failed: {e}")
    raise

Testing Groq API key...
Groq API key test successful: Hello. How can I help you today?


# Initialize Langfuse client for manual scoring

In [108]:
try:
    Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST
        )
    langfuse = get_client()
except Exception as e:
    print(f"Error initializing Langfuse client: {e}")
    raise

No 'langfuse_public_key' passed to decorated function, but multiple langfuse clients are instantiated in current process. Skipping tracing for this function to avoid cross-project leakage.


In [109]:
# Initialize Langfuse handler

In [110]:
try:
    langfuse_handler = CallbackHandler(
        public_key=LANGFUSE_PUBLIC_KEY
    )
except Exception as e:
    print(f"Error initializing Langfuse handler: {e}")
    raise

# Session management for tracing

In [111]:
session_id = None
def set_new_session_id():
    global session_id
    session_id = str(uuid.uuid4())

# Initialize session
set_new_session_id()

# Define state for LangGraph

In [112]:
from typing import TypedDict
class GraphState(TypedDict):
    user_input: str
    search_results: str
    analysis: str
    summary: str

# Node 1: Search Agent

Searches DuckDuckGo for information on the topic

In [113]:
@observe()
def search_agent(state: GraphState) -> GraphState:
    user_input = state["user_input"]
    search_query = f"{user_input} recent developments"

    # Update observation metadata
    langfuse.update_current_span(
        input=search_query,
        metadata={
            "organization": "Hobby", 
            "project": "My_Langfuse_Project1", 
            "agent": "search_agent"
            }
    )

    # Perform DuckDuckGo search
    try:
        with DDGS() as ddgs:
            results = ddgs.text(keywords=search_query, max_results=3)
            search_results = "\n".join([result['body'] for result in results]) if results else "No search results found."
    except Exception as e:
        print(f"Error during DuckDuckGo search: {e}")
        search_results = "Failed to retrieve search results."

    return {"user_input": user_input, "search_results": search_results}

# Node 2: Analyzer Agent

Analyzes search results to identify key points

In [114]:
@observe(as_type="generation")
def analyzer_agent(state: GraphState) -> GraphState:
    user_input = state["user_input"]
    search_results = state["search_results"]

    # Call Groq API to analyze the search results
    prompt = f"Analyze the following search results for the topic '{user_input}' and identify key points or trends:\n{search_results}"
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an AI that analyzes information and identifies key points or trends."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=200
    )

    analysis = response.choices[0].message.content

    # Update observation metadata
    langfuse.update_current_generation(
        input=f"User input: {user_input}\nSearch results: {search_results}",
        model="llama-3.3-70b-versatile",
        metadata={
            "organization": "Hobby", 
            "project": "My_Langfuse_Project1", 
            "agent": "analyzer_agent"
        },
        usage_details={
            "input": response.usage.prompt_tokens,
            "output": response.usage.completion_tokens
        }
    )
    
    return {"user_input": user_input, "search_results": search_results, "analysis": analysis}

# Node 3: Summarizer Agent

Summarizes the analysis into a concise report

In [115]:
@observe(as_type="generation")
def summarizer_agent(state: GraphState) -> GraphState:
    user_input = state["user_input"]
    analysis = state["analysis"]

    # Call Groq API to summarize the analysis
    prompt = f"Summarize the following analysis for the topic '{user_input}' into a concise research report (100 tokens max):\n{analysis}"
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an AI that summarizes information into concise reports."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=100
    )
    
    summary = response.choices[0].message.content

    # Update observation metadata
    langfuse.update_current_generation(
        input=f"User input: {user_input}\nAnalysis: {analysis}",
        model="llama-3.3-70b-versatile",
        metadata={
            "organization": "Hobby", 
            "project": "My_Langfuse_Project1", 
            "agent": "summarizer_agent"
            },
        usage_details={
            "input": response.usage.prompt_tokens,
            "output": response.usage.completion_tokens
        }
    )

    return {
        "user_input": user_input,
        "search_results": state["search_results"],
        "analysis": analysis,
        "summary": summary
    }

# Define the LangGraph workflow

In [116]:
workflow = StateGraph(GraphState)

# Add nodes for each agent
workflow.add_node("search_agent", search_agent)
workflow.add_node("analyzer_agent", analyzer_agent)
workflow.add_node("summarizer_agent", summarizer_agent)

# Define edges: sequential flow from search → analyzer → summarizer
workflow.set_entry_point("search_agent")
workflow.add_edge("search_agent", "analyzer_agent")
workflow.add_edge("analyzer_agent", "summarizer_agent")
workflow.add_edge("summarizer_agent", END)

# Compile the graph with Langfuse tracing

In [117]:
app = workflow.compile()
app = app.with_config({"callbacks": [langfuse_handler]})

# Main function

In [ ]:
def main():
    # Step 1: Get dynamic user input
    user_input = input("Enter a research topic (e.g., AI advancements): ").strip()
    if not user_input:
        user_input = "AI advancements"
        print("No input provided, defaulting to 'AI advancements'.")

    print(f"Researching topic: {user_input}")

    # Start a Langfuse trace for the full workflow so every agent step is captured together.
    with langfuse.start_as_current_observation(
        as_type="span",
        name="multi-agent-research-langgraph"
    ) as span:
        
        # Step 3: Use propagate_attributes to assign trace-level keys safely
        with propagate_attributes(
            user_id="demo-user-001",
            session_id=session_id
        ):
            try:
                # Execute the LangGraph workflow inside the active trace so all agent calls are linked.
                result = app.invoke(
                    {"user_input": user_input},
                    config={
                        "metadata": {
                            "session_id": session_id,
                            "organization": "Hobby",
                            "project": "My_Langfuse_Project1"
                        }
                    }
                )
                print("Search Results:", result["search_results"])
                print("Analysis:", result["analysis"])
                print("Research Summary:", result["summary"])
            except Exception as e:
                print(f"Error during graph invocation: {e}")
                result = {"summary": "Failed to generate research summary due to an error."}
                print("Research Summary:", result["summary"])

            # Step 4: Update the metadata for this span execution
            span.update(
                metadata={
                    "organization": "Hobby",
                    "project": "My_Langfuse_Project1",
                    "input_tokens": len(user_input.split()),
                    "output_tokens": len(result["summary"].split())
                }
            )

        # Step 5: Attach an evaluation score using the modern API 
        span.score(
            name="response-quality",
            value=0.9,
            comment="The research summary was concise and informative."
        )

    # Ensure all events flush out of memory before script closes
    langfuse.flush()

# Run the demo
if __name__ == "__main__":
    main()

# Instructions to view the trace in Langfuse Dashboard
print("Go to the Langfuse dashboard to view the trace:")
print("URL: https://cloud.langfuse.com")
print("Organization: Hobby")
print("Project: My_Langfuse_Project1")
print("Look under the 'Traces' tab for 'multi-agent-research-langgraph' and check 'Sessions' for session tracking.")

No 'langfuse_public_key' passed to decorated function, but multiple langfuse clients are instantiated in current process. Skipping tracing for this function to avoid cross-project leakage.


No input provided, defaulting to 'AI advancements'.
Researching topic: AI advancements


C:\Users\Shubham\AppData\Local\Temp\ipykernel_4852\2838849654.py:18: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
No 'langfuse_public_key' passed to decorated function, but multiple langfuse clients are instantiated in current process. Skipping tracing for this function to avoid cross-project leakage.
No 'langfuse_public_key' passed to decorated function, but multiple langfuse clients are instantiated in current process. Skipping tracing for this function to avoid cross-project leakage.


Search Results: No search results found.
Analysis: It appears that there are no search results available for the topic 'AI advancements'. This could be due to various reasons such as:

1. **Empty database or index**: The search database or index might be empty, or not properly populated with relevant information on AI advancements.
2. **Limited search scope**: The search scope might be too narrow, or the search query might not be specific enough to yield relevant results.
3. **Technical issues**: There could be technical issues with the search engine or the platform providing the search results, preventing relevant information from being retrieved.

In this case, I couldn't identify any key points or trends related to AI advancements. To gather meaningful insights, I would recommend:

1. **Expanding the search scope**: Broaden the search query or scope to include related topics or subfields of AI.
2. **Using alternative search engines or databases**: Try searching on different platform